In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import math, gc, os
import pyarrow as pa, pyarrow.parquet as pq

C:\Users\filipe.figueira\OneDrive - SEF-MG\Área de Trabalho\MBA\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Geração de embeddings semânticos com SBERT

In [2]:
# Define a função que gera e salva os embeddings:

def gerar_e_salvar_embeddings(
    df: pd.DataFrame,
    model: SentenceTransformer,
    text_column: str,
    ncm_code: str,
    output_dir: str,
    chunk_size: int = 2500,
    batch_size: int = 16
):
    
    """
    Gera e salva embeddings para uma coluna de texto específica, salvando os vetores
    em seu formato float32 original para máxima precisão.

    Args:
        df (pd.DataFrame): DataFrame contendo os dados.
        model (SentenceTransformer): Modelo de embedding pré-carregado.
        text_column (str): Nome da coluna com o texto a ser processado.
        ncm_code (str): Código do NCM (ex: '8703') para nomear o arquivo.
        output_dir (str): Diretório onde o arquivo Parquet será salvo.
        chunk_size (int): Número de linhas a serem processadas por bloco.
        batch_size (int): Tamanho do lote para o modelo de encoding.
    """
    
    print(f"\n--- Iniciando geração para a coluna: {text_column} (NCM: {ncm_code}) ---")
    
    output_path = os.path.join(output_dir, f"embedding_{text_column}_{ncm_code}.parquet")
    embedding_dim = model.get_sentence_embedding_dimension()
    writer = None
    
    chunk_ranges = range(0, len(df), chunk_size)
    
    for start in tqdm(chunk_ranges, desc = f"Processando '{text_column}'", unit = 'bloco', ncols = 100):
        end = min(start + chunk_size, len(df))
        block_df = df.iloc[start:end]
        
        # 1. Gera os embeddings:
        embeddings = model.encode(
            block_df[text_column].fillna('').tolist(),
            batch_size = batch_size,
            convert_to_numpy = True,
            show_progress_bar = False
        )
        
        # 2. Achata a matriz 2D de embeddings para um array 1D para o PyArrow:
        flat_embeddings = embeddings.ravel()
        
        # 3. Cria o FixedSizeListArray do PyArrow usando o tipo float32:
        embedding_array = pa.FixedSizeListArray.from_arrays(
            pa.array(flat_embeddings, type = pa.float32()),
            list_size = embedding_dim
        )
        
        # 4. Cria a tabela do PyArrow:
        table = pa.table({
            "idx": pa.array(block_df["idx"].values, type = pa.int32()),
            f"embedding_{text_column}": embedding_array
        })
        
        # 5. Escreve o bloco no arquivo Parquet:
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression='zstd')
        writer.write_table(table)
        
        del embeddings, flat_embeddings, table, embedding_array
        gc.collect()
        
    if writer:
        writer.close()
        
    print(f"✅ Embeddings de '{text_column}' salvos com sucesso em: {output_path}")

In [3]:
# ==============================================================================
# BLOCO DE EXECUÇÃO PRINCIPAL
# ==============================================================================

if __name__ == "__main__":
    
    # --- CONFIGURE SEU EXPERIMENTO AQUI ---
    CONFIG = {
        "NCM_CODE": '8708',  # Mude para '8708' para o outro dataset
        "DATA_FILE": 'dados_pre_processados_8708.csv', # Altere para o seu arquivo de dados de entrada
        "MODEL_PATH": r'Modelos/paraphrase-MiniLM-L6-v2', # Modelos/paraphrase-MiniLM-L6-v2
        "OUTPUT_DIR": 'embeddings_parquet',
        "COLUMNS_TO_PROCESS": ['xprod', 'T1', 'T2', 'T3', 'T4']
    }
    # ------------------------------------

    # Garante que o diretório de saída exista
    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True) 
    
    print("Carregando modelo SentenceTransformer...")
    model = SentenceTransformer(CONFIG["MODEL_PATH"], device = 'cpu')
    
    print(f"Carregando e preparando dados de: {CONFIG['DATA_FILE']}")
    df = pd.read_csv(CONFIG["DATA_FILE"])
    df.reset_index(drop = True, inplace = True)
    df['idx'] = df.index
    
    for col in CONFIG["COLUMNS_TO_PROCESS"]:
        gerar_e_salvar_embeddings(
            df = df,
            model = model,
            text_column = col,
            ncm_code = CONFIG["NCM_CODE"],
            output_dir = CONFIG["OUTPUT_DIR"]
        )
    
    print("\nProcesso de geração de embeddings concluído com sucesso!")

Carregando modelo SentenceTransformer...
Carregando e preparando dados de: dados_pre_processados_8708.csv

--- Iniciando geração para a coluna: xprod (NCM: 8708) ---


Processando 'xprod': 100%|███████████████████████████████████████| 20/20 [03:30<00:00, 10.53s/bloco]


✅ Embeddings de 'xprod' salvos com sucesso em: embeddings_parquet\embedding_xprod_8708.parquet

--- Iniciando geração para a coluna: T1 (NCM: 8708) ---


Processando 'T1': 100%|██████████████████████████████████████████| 20/20 [03:00<00:00,  9.02s/bloco]


✅ Embeddings de 'T1' salvos com sucesso em: embeddings_parquet\embedding_T1_8708.parquet

--- Iniciando geração para a coluna: T2 (NCM: 8708) ---


...

✅ Embeddings de 'T2' salvos com sucesso em: embeddings_parquet\embedding_T2_8708.parquet

--- Iniciando geração para a coluna: T3 (NCM: 8708) ---


Processando 'T3': 100%|██████████████████████████████████████████| 20/20 [02:50<00:00,  8.52s/bloco]


✅ Embeddings de 'T3' salvos com sucesso em: embeddings_parquet\embedding_T3_8708.parquet

--- Iniciando geração para a coluna: T4 (NCM: 8708) ---


Processando 'T4': 100%|██████████████████████████████████████████| 20/20 [02:13<00:00,  6.65s/bloco]

✅ Embeddings de 'T4' salvos com sucesso em: embeddings_parquet\embedding_T4_8708.parquet

Processo de geração de embeddings concluído com sucesso!


# Gerando embeddings semânticos com SBERT multilíngue:

In [4]:
# ==============================================================================
# BLOCO DE EXECUÇÃO PRINCIPAL
# ==============================================================================

if __name__ == "__main__":
    
    # --- CONFIGURE SEU EXPERIMENTO AQUI ---
    CONFIG = {
        "NCM_CODE": '8708',  # Mude para '8708' para o outro dataset
        "DATA_FILE": 'dados_pre_processados_8708.csv', # Altere para o seu arquivo de dados de entrada
        "MODEL_PATH": r'Modelos/paraphrase-multilingual-MiniLM-L12-v2', # Modelos/paraphrase-MiniLM-L6-v2
        "OUTPUT_DIR": 'embeddings_parquet_multilingual',
        "COLUMNS_TO_PROCESS": ['xprod', 'T1', 'T2', 'T3', 'T4']
    }
    # ------------------------------------

    # Garante que o diretório de saída exista
    os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True) 
    
    print("Carregando modelo SentenceTransformer...")
    model = SentenceTransformer(CONFIG["MODEL_PATH"], device = 'cpu')
    
    print(f"Carregando e preparando dados de: {CONFIG['DATA_FILE']}")
    df = pd.read_csv(CONFIG["DATA_FILE"])
    df.reset_index(drop = True, inplace = True)
    df['idx'] = df.index
    
    for col in CONFIG["COLUMNS_TO_PROCESS"]:
        gerar_e_salvar_embeddings(
            df = df,
            model = model,
            text_column = col,
            ncm_code = CONFIG["NCM_CODE"],
            output_dir = CONFIG["OUTPUT_DIR"]
        )
    
    print("\nProcesso de geração de embeddings concluído com sucesso!")

Carregando modelo SentenceTransformer...
Carregando e preparando dados de: dados_pre_processados_8708.csv

--- Iniciando geração para a coluna: xprod (NCM: 8708) ---


Processando 'xprod': 100%|███████████████████████████████████████| 20/20 [05:48<00:00, 17.41s/bloco]


✅ Embeddings de 'xprod' salvos com sucesso em: embeddings_parquet_multilingual\embedding_xprod_8708.parquet

--- Iniciando geração para a coluna: T1 (NCM: 8708) ---


Processando 'T1': 100%|██████████████████████████████████████████| 20/20 [05:18<00:00, 15.91s/bloco]


✅ Embeddings de 'T1' salvos com sucesso em: embeddings_parquet_multilingual\embedding_T1_8708.parquet

--- Iniciando geração para a coluna: T2 (NCM: 8708) ---


...

✅ Embeddings de 'T2' salvos com sucesso em: embeddings_parquet_multilingual\embedding_T2_8708.parquet

--- Iniciando geração para a coluna: T3 (NCM: 8708) ---


Processando 'T3': 100%|██████████████████████████████████████████| 20/20 [04:43<00:00, 14.18s/bloco]


✅ Embeddings de 'T3' salvos com sucesso em: embeddings_parquet_multilingual\embedding_T3_8708.parquet

--- Iniciando geração para a coluna: T4 (NCM: 8708) ---


Processando 'T4': 100%|██████████████████████████████████████████| 20/20 [03:57<00:00, 11.90s/bloco]

✅ Embeddings de 'T4' salvos com sucesso em: embeddings_parquet_multilingual\embedding_T4_8708.parquet

Processo de geração de embeddings concluído com sucesso!


In [5]:
df.to_csv('dados_pre_processados_8708.csv')